In [1]:
import os
import torch, torchvision
import torch.nn as nn
from torchvision import transforms
from PIL import Image
from models.ctran import ctranspath
from models.CHIEF import CHIEF
import general_fcns as gf
import pandas as pd
import numpy as np


c:\Users\miskovicvanja\.conda\envs\eric_env\Lib\site-packages\timm\models\layers\__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
import torch
print(torch.version.cuda)  # This will show the CUDA version that PyTorch was built with


None


In [3]:
# Paths
image_folder = './eric/data_pdl1'

patch_model_path = './model_weight/CHIEF_CTransPath.pth'
wsi_model_path = './model_weight/CHIEF_pretraining.pth'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [4]:
import models.ctran
print(models.ctran.__file__)


c:\Users\miskovicvanja\PDL1_FM_DP_2025_ERIC_LOMBARDO_FACCIALE\Eric\Benchmarking-for-Foundation-Model-pdl1-chief\models\ctran.py


In [5]:
from models.ctran import ConvStem  # <--- explicitly import YOUR class


In [6]:
mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)
transform = transforms.Compose([
    transforms.Resize((224, 224)),   # Force resize to exactly (224,224)
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])



model = ctranspath()
model.head = nn.Identity()
td = torch.load(r'./model_weight/CHIEF_CTransPath.pth')
model.load_state_dict(td['model'], strict=False)
model.eval()


c:\Users\miskovicvanja\PDL1_FM_DP_2025_ERIC_LOMBARDO_FACCIALE\Eric\eric_env2\Lib\site-packages\torch\functional.py:512: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ..\aten\src\ATen\native\TensorShape.cpp:3588.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


SwinTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
    (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (layers): Sequential(
    (0): BasicLayer(
      dim=96, input_resolution=(56, 56), depth=2
      (blocks): ModuleList(
        (0): SwinTransformerBlock(
          (norm1): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
          (attn): WindowAttention(
            (qkv): Linear(in_features=96, out_features=288, bias=True)
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=96, out_features=96, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
            (softmax): Softmax(dim=-1)
          )
          (drop_path): Identity()
          (norm2): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Linear(in_features=96, out_features=384, bias=True)
            (a

In [7]:
# Patch-level model
patch_model = ctranspath()
patch_model.head = nn.Identity()
td_patch = torch.load(patch_model_path, map_location=device)
patch_model.load_state_dict(td_patch['model'], strict=False)
patch_model = patch_model.to(device)
patch_model.eval()

# WSI-level model
wsi_model = CHIEF(size_arg="small", dropout=True, n_classes=2)
td_wsi = torch.load(wsi_model_path, map_location=device)
wsi_model.load_state_dict(td_wsi, strict=True)
wsi_model = wsi_model.to(device)
wsi_model.eval()

[768, 512, 256]


CHIEF(
  (attention_net): Sequential(
    (0): Linear(in_features=768, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.25, inplace=False)
    (3): Attn_Net_Gated(
      (attention_a): Sequential(
        (0): Linear(in_features=512, out_features=256, bias=True)
        (1): Tanh()
        (2): Dropout(p=0.25, inplace=False)
      )
      (attention_b): Sequential(
        (0): Linear(in_features=512, out_features=256, bias=True)
        (1): Sigmoid()
        (2): Dropout(p=0.25, inplace=False)
      )
      (attention_c): Linear(in_features=256, out_features=1, bias=True)
    )
  )
  (classifiers): Linear(in_features=512, out_features=2, bias=True)
  (instance_classifiers): ModuleList(
    (0-1): 2 x Linear(in_features=512, out_features=2, bias=True)
  )
  (instance_loss_fn): CrossEntropyLoss()
  (att_head): Att_Head(
    (fc1): Linear(in_features=512, out_features=256, bias=True)
    (relu): ReLU()
    (fc2): Linear(in_features=256, out_features=1, bias=True)
    (s

In [8]:
import os
import pandas as pd
from PIL import Image
import torch

# Get the current working directory and the CSV file path
csv_file_path = os.path.join(os.getcwd(), 'filtered_metadata_full_path.csv')
skip = ["D:/Digital_path_unzipped/SZMC1050237_pdl1.ndpi", "D:/Digital_path/DIG_PAT_1727366215.ndpi"]

# Check if the CSV file exists
if not os.path.exists(csv_file_path):
    print(f"CSV file not found: {csv_file_path}")
else:
    # Load CSV file containing the list of image paths (no header)
    df = pd.read_csv(csv_file_path, header=None)  # Read CSV without assuming header
    
    total_slides = len(df)
    
    # Loop over each image path in the CSV file with progress info
    for idx, img_path in enumerate(df[0], start=1):  # 1-based index for nicer display
        img_path = img_path.strip()  # Remove any leading/trailing whitespace
        img_file = os.path.basename(img_path)
        
        if img_path in skip:
            continue
        
        print(f"Processing slide {idx}/{total_slides}: {img_file}")
        
        if os.path.exists(img_path):  # Check if file exists before processing
            
            # Check file extension and process accordingly
            if img_file.lower().endswith(('.svs', '.tif', '.ndpi')):
                slide = gf.slide_at_magnification(img_path, magnification_params={'magnification': 10})
                image = Image.fromarray(slide)
            elif img_file.lower().endswith(('.png', '.jpg', '.jpeg')):
                image = Image.open(img_path).convert('RGB')
            else:
                # Handling for unsupported file types
                print(f"Skipping unsupported file type: {img_file}")
                continue  # Skip this file and move to the next one
            
            image = transform(image).unsqueeze(0).to(device)
            
            # --- Get patch feature ---
            with torch.no_grad():
                patch_feature = patch_model(image)  # [1, 768]
            
            # Set anatomical site
            anatomical_site_idx = 13  # <-- change depending on your case
            
            # --- Get WSI-level feature ---
            with torch.no_grad():
                x = patch_feature
                anatomical = torch.tensor([anatomical_site_idx], device=device)
                output = wsi_model(x, anatomical)
                wsi_feature_emb = output['WSI_feature']  # [1, 768]
            
            print(f"WSI-level feature for {img_file}: {wsi_feature_emb.shape}")
            
            # Save feature to ./features/
            os.makedirs('./features/', exist_ok=True)
            save_path = os.path.join('./features/', img_file.replace('.', '_') + '_feature.pt')
            torch.save(wsi_feature_emb.cpu(), save_path)
            print(f"Saved feature to {save_path}")
        else:
            print(f"Image path {img_path} does not exist.")


Processing slide 1/472: DIG_PAT_1701099612.tif
WSI-level feature for DIG_PAT_1701099612.tif: torch.Size([1, 768])
Saved feature to ./features/DIG_PAT_1701099612_tif_feature.pt
Processing slide 2/472: DIG_PAT_1701101606.tif
WSI-level feature for DIG_PAT_1701101606.tif: torch.Size([1, 768])
Saved feature to ./features/DIG_PAT_1701101606_tif_feature.pt
Processing slide 3/472: DIG_PAT_1701103165.tif
WSI-level feature for DIG_PAT_1701103165.tif: torch.Size([1, 768])
Saved feature to ./features/DIG_PAT_1701103165_tif_feature.pt
Processing slide 4/472: DIG_PAT_1701103988.tif
WSI-level feature for DIG_PAT_1701103988.tif: torch.Size([1, 768])
Saved feature to ./features/DIG_PAT_1701103988_tif_feature.pt
Processing slide 5/472: DIG_PAT_1701104914.tif
WSI-level feature for DIG_PAT_1701104914.tif: torch.Size([1, 768])
Saved feature to ./features/DIG_PAT_1701104914_tif_feature.pt
Processing slide 6/472: DIG_PAT_1701121130.tif
WSI-level feature for DIG_PAT_1701121130.tif: torch.Size([1, 768])
Saved 